# LIKAS — GGUF Export (run in a CLEAN Kaggle session)

Companion to the training notebook. That notebook fine-tunes `gemma-4-e2b-it` with Unsloth and pushes the **LoRA adapter** to `jpcurada/likas-gemma4-e2b-lora`. This notebook produces the on-device artifact: a 4-bit `GGUF`.

### How it works (and why this exact path)

Gemma 4 E2B is **multimodal** (text + audio + vision towers). Through hard testing we established:

- `convert_lora_to_gguf.py` and `llama-export-lora` **cannot** handle the multimodal base (`audio_tower` tensors are unmappable). Dead end.
- `convert_hf_to_gguf.py` **does** handle it — it cleanly produced a 601-tensor text GGUF of the base.
- PEFT/torch merging is broken here: the llama.cpp converter requirements downgrade torch, which shatters the `torch → triton → torchao → transformers → peft` import chain.

**So the only sound path:** merge the LoRA into the base with **pure numpy + safetensors** (zero torch/peft — immune to the broken chain), then run the **proven** `convert_hf_to_gguf.py`, then quantize.

### Disk model

`/kaggle/working` is ~20 GB; the fp16 model is ~10 GB. We keep the base and the merged HF model in **`/tmp`** (a separate, larger filesystem) so only the final GGUFs touch the working quota.

### Before you run

1. **Fresh** Kaggle notebook, **GPU** on (T4 fine — only needed so the env matches; the pipeline is CPU).
2. **Add-ons → Secrets**: add `HF_TOKEN` (Hugging Face *write* token).
3. `/kaggle/working` must start empty.
4. Run top to bottom. Steps are guarded — a re-run resumes instead of repeating slow work.

## I. Setup

In [ ]:
%%capture
# Minimal deps. The merge is pure numpy+safetensors (no torch/peft), so we do
# NOT install the training stack. transformers is pinned to 5.5.0 because the
# llama.cpp converter's tokenizer step needs a Gemma-4-aware version, and it is
# installed --no-deps so nothing drags torch/torchvision back in.
import subprocess

!pip install -q -U "huggingface_hub>=1.5.0,<2.0" safetensors numpy sentencepiece protobuf
!pip install -q --no-deps "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0"

# torchvision/torchaudio, if present, are built for a different torch than the
# converter pulls; their broken C++ ABI makes transformers mis-report a
# 'Gemma4Config' import error. Remove them — nothing here needs vision/audio.
!pip uninstall -y -q torchvision torchaudio

# Verify in a FRESH SUBPROCESS — that is exactly how convert_hf_to_gguf.py runs.
_chk = subprocess.run(
    'python -c "from transformers import Gemma4Config; '
    'import transformers; print(transformers.__version__)"',
    shell=True, capture_output=True, text=True)
assert _chk.stdout.strip() and "Traceback" not in _chk.stderr, (
    "Subprocess Gemma4Config import FAILED:\n" + _chk.stderr[-1500:])
print("Subprocess transformers", _chk.stdout.strip(), "— Gemma4Config OK")

In [ ]:
import os, glob, json, shutil, subprocess
import numpy as np
from safetensors import safe_open
from safetensors.numpy import save_file
from huggingface_hub import HfApi, login, create_repo, snapshot_download

print("Imports OK. numpy", np.__version__)

In [ ]:
# --- Constants ---
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

WORKDIR     = "/kaggle/working"
HF_USER     = "jpcurada"
LORA_REPO   = f"{HF_USER}/likas-gemma4-e2b-lora"
GGUF_REPO   = f"{HF_USER}/likas-gemma4-e2b-gguf"
GGUF_PATH   = f"{WORKDIR}/likas-q4_k_m.gguf"
MODEL_CARD_PATH = "/kaggle/input/datasets/jeypiic/likas-ai-datasets/MODEL_CARD.md"

BASE_DIR    = "/tmp/base_fp16"      # HF base, /tmp (separate fs from the quota)
MERGED_HF   = "/tmp/merged_hf"      # merged HF model, also /tmp
MERGED_GGUF = f"{WORKDIR}/merged-f16.gguf"

def disk_free_gb(p=WORKDIR):
    _, _, f = shutil.disk_usage(p)
    return round(f / 1024**3, 2)

def sh(cmd):
    print("+", cmd)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

print(f"Free disk: {disk_free_gb()} GB")

## II. Fetch the LoRA adapter and the fp16 base

The adapter (~250 MB) goes to the HF cache; the base (~10 GB) goes to `/tmp` (not the working quota). Both have a `skip if present` guard so a re-run doesn't re-download.

In [ ]:
local_lora = snapshot_download(repo_id=LORA_REPO, token=HF_TOKEN)
print("LoRA adapter at:", local_lora)
print("Files:", sorted(os.listdir(local_lora)))

if not os.path.exists(f"{BASE_DIR}/config.json"):
    print(f"\nDownloading fp16 base to {BASE_DIR} ...")
    snapshot_download(
        repo_id="unsloth/gemma-4-e2b-it", local_dir=BASE_DIR, token=HF_TOKEN,
        allow_patterns=["*.safetensors", "*.json", "*.model", "tokenizer*"],
    )
print(f"Base ready. Working free: {disk_free_gb()} GB")

## III. Merge the LoRA into the base — pure numpy, no torch/peft

For every adapted weight: `W_merged = W_base + (B @ A) * (alpha / r)`.

This reads/writes safetensors with numpy only — it never imports torch, peft, transformers, torchao, or triton, so the broken-torch dependency chain cannot touch it. We copy the base shard-by-shard, applying the LoRA delta to the targeted projection weights, and keep the base's index + config + tokenizer so the result is a valid HF model directory.

In [ ]:
import struct

# NumPy has no bfloat16. safetensors files store dtype in a JSON header; we
# read raw bytes and upconvert bf16 -> fp32 ourselves (bf16 = high 16 bits of
# fp32, so: widen to uint32, shift left 16, reinterpret as float32 — exact).

def _read_st_header(path):
    with open(path, "rb") as fh:
        (n,) = struct.unpack("<Q", fh.read(8))
        return json.loads(fh.read(n)), 8 + n

def _load_tensor(path, header, data_off, name):
    info = header[name]
    dt = info["dtype"]
    s, e = info["data_offsets"]
    with open(path, "rb") as fh:
        fh.seek(data_off + s)
        raw = fh.read(e - s)
    shape = info["shape"]
    if dt in ("F32", "float32"):
        arr = np.frombuffer(raw, dtype="<f4")
    elif dt in ("F16", "float16"):
        arr = np.frombuffer(raw, dtype="<f2").astype(np.float32)
    elif dt in ("BF16", "bfloat16"):
        u16 = np.frombuffer(raw, dtype="<u2").astype(np.uint32)
        arr = (u16 << 16).view(np.float32)
    else:
        raise ValueError(f"Unhandled dtype {dt} for {name}")
    return arr.reshape(shape), dt

def _fp32_to_fp16_bytes(arr):
    return np.ascontiguousarray(arr.astype(np.float16))

if not os.path.exists(f"{MERGED_HF}/config.json"):
    os.makedirs(MERGED_HF, exist_ok=True)

    acfg = json.load(open(os.path.join(local_lora, "adapter_config.json")))
    r, alpha = acfg["r"], acfg["lora_alpha"]
    scaling = alpha / r
    fan_in_fan_out = acfg.get("fan_in_fan_out", False)
    print(f"LoRA r={r} alpha={alpha} scaling={scaling} fan_in_fan_out={fan_in_fan_out}")

    # Load LoRA tensors (they are fp32 in this adapter -> numpy reads fine).
    lora_st = os.path.join(local_lora, "adapter_model.safetensors")
    lhdr, ldoff = _read_st_header(lora_st)
    lhdr = {k: v for k, v in lhdr.items() if k != "__metadata__"}
    pairs = {}
    for k in lhdr:
        if ".lora_A." in k or ".lora_B." in k:
            base_name = (k.replace("base_model.model.", "")
                          .replace(".lora_A.weight", ".weight")
                          .replace(".lora_B.weight", ".weight"))
            arr, _ = _load_tensor(lora_st, lhdr, ldoff, k)
            pairs.setdefault(base_name, {})["A" if ".lora_A." in k else "B"] = arr
    pairs = {n: v for n, v in pairs.items() if "A" in v and "B" in v}
    print(f"{len(pairs)} base weights to merge (incl. audio_tower — expected).")

    shards = sorted(glob.glob(f"{BASE_DIR}/*.safetensors"))
    assert shards, f"No base safetensors in {BASE_DIR}"
    merged = 0
    for shard in shards:
        hdr, doff = _read_st_header(shard)
        meta = hdr.get("__metadata__", {"format": "pt"})
        hdr = {k: v for k, v in hdr.items() if k != "__metadata__"}
        out = {}
        for name in hdr:
            W, _dt = _load_tensor(shard, hdr, doff, name)        # fp32 view
            if name in pairs:
                A = pairs[name]["A"].astype(np.float32)          # [r, in]
                B = pairs[name]["B"].astype(np.float32)          # [out, r]
                delta = (B @ A) * scaling
                if fan_in_fan_out:
                    delta = delta.T
                W = W + delta
                merged += 1
            # Store as fp16 (what convert_hf_to_gguf --outtype f16 expects).
            out[name] = _fp32_to_fp16_bytes(W)
        save_file(out, os.path.join(MERGED_HF, os.path.basename(shard)),
                  metadata=meta)
        del out
        print(f"  {os.path.basename(shard)} done | working free {disk_free_gb()} GB")

    assert merged == len(pairs), (
        f"Merged {merged} but expected {len(pairs)} — name mapping is off.")

    # Copy non-weight files the converter needs (config, tokenizer, etc.).
    # Skip directories (e.g. snapshot_download's .cache) and weights.
    def _copy_aux(src_dir):
        for fn in os.listdir(src_dir):
            sp = os.path.join(src_dir, fn)
            if os.path.isdir(sp) or fn.endswith(".safetensors"):
                continue
            shutil.copy2(sp, os.path.join(MERGED_HF, fn))

    _copy_aux(BASE_DIR)
    # Prefer the fine-tune's tokenizer/chat template if it shipped one.
    for fn in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja",
               "special_tokens_map.json"):
        src = os.path.join(local_lora, fn)
        if os.path.exists(src) and os.path.isfile(src):
            shutil.copy2(src, os.path.join(MERGED_HF, fn))
    print(f"\nMerged {merged} weights (fp16) -> {MERGED_HF}")
    print("Merged dir contents:", sorted(os.listdir(MERGED_HF)))

print(f"Merged HF ready. Working free: {disk_free_gb()} GB")

## IV. Convert + quantize with llama.cpp

Build `llama.cpp`, run the **proven** `convert_hf_to_gguf.py` on the merged model (it already handled this exact architecture for the base), free `/tmp`, then `llama-quantize` → **Q4_K_M**. Only the ~1.8 GB final GGUF stays in `/kaggle/working`.

In [ ]:
# Build llama.cpp (CPU). Scrub transformers/tokenizers from its requirements
# (edited in place — they `-r`-include siblings by relative path) so the pinned
# transformers==5.5.0 survives; then re-drop torchvision/torchaudio.
if not os.path.exists("/tmp/llama.cpp/build/bin/llama-quantize"):
    sh("rm -rf /tmp/llama.cpp")
    sh("git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp")
    rq = "/tmp/llama.cpp/requirements"
    for fn in os.listdir(rq):
        if not fn.endswith(".txt"):
            continue
        p = os.path.join(rq, fn)
        lines = open(p).readlines()
        with open(p, "w") as f:
            for ln in lines:
                lo = ln.strip().lower()
                f.write(f"# scrubbed: {ln}" if (lo.startswith("transformers")
                        or lo.startswith("tokenizers")) else ln)
    sh("pip install -q -r /tmp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")
    sh("pip uninstall -y -q torchvision torchaudio")
    sh("cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF "
       "-DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release >/dev/null 2>&1")
    sh("cmake --build /tmp/llama.cpp/build -j --target llama-quantize >/dev/null 2>&1")
QUANT = "/tmp/llama.cpp/build/bin/llama-quantize"

# Re-verify Gemma4Config in a subprocess after the converter deps install.
_chk = subprocess.run(
    'python -c "from transformers import Gemma4Config; '
    'import transformers; print(transformers.__version__)"',
    shell=True, capture_output=True, text=True)
if "Traceback" in _chk.stderr or not _chk.stdout.strip():
    raise RuntimeError("Subprocess Gemma4Config import failed:\n" + _chk.stderr[-1500:])
print("Subprocess transformers", _chk.stdout.strip(), "— Gemma4Config OK")

# Merged HF -> fp16 GGUF (the converter proven to handle Gemma 4 multimodal).
if not os.path.exists(MERGED_GGUF):
    sh(f"python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED_HF} "
       f"--outfile {MERGED_GGUF} --outtype f16")
print(f"Merged GGUF ready. Working free: {disk_free_gb()} GB")

# Free /tmp, then quantize -> Q4_K_M and drop the fp16.
shutil.rmtree(BASE_DIR, ignore_errors=True)
shutil.rmtree(MERGED_HF, ignore_errors=True)
sh(f"{QUANT} {MERGED_GGUF} {GGUF_PATH} q4_k_m")
os.remove(MERGED_GGUF)

print(f"\nGGUF written: {GGUF_PATH} | working free: {disk_free_gb()} GB")
sh(f"ls -lh {GGUF_PATH}")

## V. Smoke test the GGUF

Load the quantized GGUF through `llama-cpp-python` — the same llama.cpp engine the phone uses via `llama.rn`. A valid single-JSON reply means it behaves the same offline on-device.

In [ ]:
!pip install -q llama-cpp-python
from llama_cpp import Llama

llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_gpu_layers=-1, verbose=False)
out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are LIKAS, an offline disaster "
         "assistant for the Philippines. Reply with exactly one JSON object: "
         "a tool call or a message to the user."},
        {"role": "user", "content": "May lindol! Ano gagawin ko ngayon?"},
    ],
    max_tokens=96, temperature=0.0, stop=["<end_of_turn>"],
)
print("Smoke test reply:")
print(out["choices"][0]["message"]["content"])

## VI. Push the GGUF to Hugging Face

In [ ]:
login(token=HF_TOKEN)
api = HfApi()
create_repo(GGUF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)

api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo="likas-q4_k_m.gguf",
                repo_id=GGUF_REPO, token=HF_TOKEN)

if os.path.exists(MODEL_CARD_PATH):
    api.upload_file(path_or_fileobj=MODEL_CARD_PATH, path_in_repo="README.md",
                    repo_id=GGUF_REPO, token=HF_TOKEN)
    print("README uploaded.")
else:
    print(f"No model card at {MODEL_CARD_PATH} — skipped (harmless).")

print(f"\nGGUF live: https://huggingface.co/{GGUF_REPO}")